# KHOMA Quant Research Lab - Research Walkthrough

This notebook walks through the whole research workflow, step by step, using the same functions as the dashboard:

1. Load and check historical data
2. Returns, moving averages and volatility
3. The three strategies (target weights)
4. Backtesting - and why the timing matters
5. Performance and risk metrics
6. In-sample vs out-of-sample, and overfitting
7. Transaction-cost sensitivity
8. Multi-asset portfolios and diversification
9. Stress testing
10. Exporting a report

**Run the cells from top to bottom** (in PyCharm: right-click the notebook and choose *Run All*, or press Shift+Enter cell by cell).

In [ ]:
import sys
from pathlib import Path

# Make "import src..." work when the notebook runs from the notebooks/ folder
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import matplotlib.pyplot as plt

from src.data_loader import load_prices
from src.indicators import simple_moving_average, rolling_volatility
from src.strategies import STRATEGIES, get_target_weights
from src.backtester import run_backtest, run_strategy_backtest
from src.risk_metrics import performance_summary, formatted_summary_table, drawdown_series
from src.research import chronological_split_date, train_test_report, moving_average_parameter_search, cost_sensitivity
from src.portfolio import simulate_portfolio, portfolio_as_backtest, equal_weights, correlation_matrix, risk_contributions, diversification_ratio
from src.stress_testing import stress_test
from src.visualization import static_equity_and_drawdown

pd.set_option("display.float_format", lambda v: f"{v:,.4f}")

## 1. Load data

We try Yahoo Finance first. If it can't be reached (no internet, or Yahoo changed something), the notebook switches to the bundled **synthetic** data and says so loudly. Synthetic results are only a demonstration of the code - they are not real market history.

In [ ]:
REAL_TICKERS = ["SPY", "QQQ", "IWM", "TLT", "GLD"]
SYNTHETIC_TICKERS = ["SYN_STOCKS", "SYN_TECH", "SYN_SMALLCAP", "SYN_BONDS", "SYN_GOLD"]
START, END = "2005-01-01", "2024-12-31"

prices, info = load_prices(REAL_TICKERS, START, END, source="yahoo")
if prices.empty or len(prices.columns) < len(REAL_TICKERS):
    print("Yahoo Finance data unavailable:", info["errors"])
    print(">>> USING SYNTHETIC DATA - results below are NOT real market performance <<<")
    prices, info = load_prices(SYNTHETIC_TICKERS, START, END, source="synthetic")
    DATA_LABEL = "SYNTHETIC"
else:
    DATA_LABEL = "Yahoo Finance"

STOCK, BOND = prices.columns[0], prices.columns[3]
print(f"Data: {DATA_LABEL} | {prices.index.min().date()} to {prices.index.max().date()} | {len(prices)} days")
prices.tail()

In [ ]:
# Data quality: what did cleaning find and remove? (Nothing is ever forward-filled.)
pd.DataFrame({t: {k: r[k] for k in ["rows", "duplicate_dates", "missing_prices", "rows_removed"]}
              for t, r in info["reports"].items()}).T

## 2. Returns, moving averages and volatility

* Daily return: $r_t = P_t / P_{t-1} - 1$
* A 50-day simple moving average is the mean of the last 50 closes **including today** - it never uses tomorrow's price.
* Annualized volatility = daily standard deviation $\times \sqrt{252}$.

In [ ]:
price = prices[STOCK]
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(price, label=f"{STOCK} adjusted close", linewidth=1.2)
ax.plot(simple_moving_average(price, 50), label="50-day SMA", linewidth=1)
ax.plot(simple_moving_average(price, 200), label="200-day SMA", linewidth=1)
ax.set_title(f"{STOCK} with moving averages ({DATA_LABEL} data)")
ax.legend(); plt.show()

rolling_volatility(price, 63).describe()

## 3. Strategies produce target weights

A strategy only answers: *"after today's close, what fraction of the portfolio do I want in the asset?"* It never calculates profit.

In [ ]:
weights = pd.DataFrame({name: get_target_weights(name, price) for name in STRATEGIES})
weights.iloc[195:205]   # around day 200, where the 200-day average first exists

## 4. Backtesting and timing

The default execution model: signal after the close of day *t* -> trade at the close of day *t+1* -> earn the return of day *t+2*. Watch the `target_weight` and `weight` columns: the weight actually held lags the target by two rows.

In [ ]:
bt = run_strategy_backtest(price, "Moving Average Trend (50/200)", initial_capital=10_000, slippage_bps=5)
first_buy = bt.index[bt["target_weight"].diff() > 0][0]
pos = bt.index.get_loc(first_buy)
bt.iloc[pos - 1: pos + 4][["price", "target_weight", "weight", "turnover", "cost", "strategy_return", "equity"]]

In [ ]:
# How much does the timing assumption matter? Compare same-close vs next-close execution.
optimistic = run_strategy_backtest(price, "Moving Average Trend (50/200)", execution_delay=0)
realistic = run_strategy_backtest(price, "Moving Average Trend (50/200)", execution_delay=1)
pd.DataFrame({"same close (optimistic)": performance_summary(optimistic),
              "next close (default)": performance_summary(realistic)}).loc[["cagr", "sharpe", "max_drawdown"]]

## 5. Performance metrics for all three strategies

In [ ]:
RISK_FREE = 0.02
backtests = {name: run_strategy_backtest(price, name, slippage_bps=5) for name in STRATEGIES}
summaries = {name: performance_summary(b, RISK_FREE) for name, b in backtests.items()}
formatted_summary_table(summaries)

In [ ]:
fig = static_equity_and_drawdown({n: b["equity"] for n, b in backtests.items()}, f"{STOCK} ({DATA_LABEL} data)")
plt.show()

## 6. In-sample vs out-of-sample

The first 60% of dates is the training period and the last 40% is the test period. The split is chronological - shuffling time series would leak future information into training.

In [ ]:
split = chronological_split_date(price.index, 0.6)
print("Test period starts:", split.date())
train_test_report(price, "Volatility-Targeted Trend", None, split, RISK_FREE).loc[["cagr", "annual_volatility", "sharpe", "max_drawdown"]]

In [ ]:
# Overfitting check: rank moving-average windows by TRAINING Sharpe, then look at TEST Sharpe.
search = moving_average_parameter_search(price, split, risk_free_rate=RISK_FREE)
search

**Questions to ask yourself:** Is the rank-1 pair also the best in the test column? How far does its Sharpe fall (or rise) out-of-sample? What would happen if you kept re-running this until the test number looked good?

## 7. Transaction-cost sensitivity

In [ ]:
cost_sensitivity(price, "Volatility-Targeted Trend", cost_levels_bps=[0, 5, 10, 25, 50], risk_free_rate=RISK_FREE)

## 8. Portfolios and diversification

Weights are fixed in advance (no returns are used to pick them), drift between rebalances, and are reset on the first trading day of each quarter, with costs charged on the turnover.

In [ ]:
portfolios = {"60/40": {STOCK: 0.6, BOND: 0.4}, "Equal weight": equal_weights(list(prices.columns))}
sims = {n: simulate_portfolio(prices[list(w)], w, rebalance="Q", cost_bps=5) for n, w in portfolios.items()}
formatted_summary_table({n: performance_summary(portfolio_as_backtest(s), RISK_FREE) for n, s in sims.items()})

In [ ]:
w = portfolios["60/40"]
print("Diversification ratio:", round(diversification_ratio(w, prices), 2))
display(risk_contributions(w, prices))     # 60% of the money, but how much of the risk?
correlation_matrix(prices).round(2)

## 9. Stress testing

Named crisis windows are only meaningful on real data. With synthetic data, the dates are just labels on random numbers, so they are skipped.

In [ ]:
if DATA_LABEL == "SYNTHETIC":
    print("Skipping named crisis windows on synthetic data.")
else:
    for name, b in backtests.items():
        table, skipped = stress_test(b["strategy_return"], b["equity"])
        print(name); display(table[["period_return", "max_drawdown", "days_to_recover_after_window"]]); print(skipped)

## 10. Export a report

In [ ]:
from src.reporting import generate_markdown_report

settings = {"Asset": STOCK, "Data": DATA_LABEL, "Slippage (bps)": 5, "Execution": "next day's close"}
report = generate_markdown_report(settings, backtests, "Buy and Hold", RISK_FREE,
                                  data_note="Synthetic data" if DATA_LABEL == "SYNTHETIC" else "")
print(report[:1500])

## Your turn

1. Change the moving-average windows in section 4 to 20/100. What happens to turnover and costs?
2. In section 7, find the cost level at which the volatility-targeted strategy's CAGR falls below buy-and-hold.
3. Add a 70/30 portfolio in section 8 and compare its maximum drawdown with 60/40.